# v12 — native tcgen05 tensor-core MLA decode — the gate (RENT root B300/sm_103a; the ENGINE change)

Run-of-record for v12: fork v11's MLA decode changing **one variable — the compute ENGINE** (warp-per-head
CUDA-core GEMV → Blackwell **tcgen05 tensor cores**: Arm 1 FP8 MMA, Arm 2 native NVFP4 MMA). Storage,
split-KV, LSE merge, score-stationary structure, and the NVFP4 latent pool are **byte-identical to v11**,
so every comparison here is a clean **engine A/B vs v11**.

**HARDWARE: a rented root/bare-metal B300 / sm_103a** (CUDA 12.9+, CUTLASS 4.x). NOT Colab/T4 — tensor-core
MLA needs Blackwell (dispatch gate `(10,0)`). A B200/sm_100 dev rung can de-risk the build; the record runs
on B300. **Pay v11's four honesty debts here:** ncu on a privileged box · lock clocks (not idle-pinned) ·
profile the torch baseline at fp16/bf16 · re-ablate the 2×-exp at M=128.

⚠️ **The kernel body is GPU-implementation work.** `kernels/v12_mla_tc/mla_tc_attention.cu` ships as a
documented SCAFFOLD (fork CUTLASS example 77; Arm 1 FP8 → Arm 2 NVFP4). Until the tcgen05 body lands, the
host entry `TORCH_CHECK(false, …)` so the build/wiring is testable and correctness fails loudly. Implement
the body (this notebook's §3 points at the file), then re-run §4 onward.


## 0. Dependencies + GPU (venv-safe)

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit('No GPU on this runtime. v12 needs a Blackwell B300/sm_103a (or a B200/sm_100 dev rung).')

pip('ninja', 'pytest', 'numpy')

try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False
if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu129'))
    raise SystemExit('torch was CPU-only -> installed the CUDA build. Restart the kernel and re-run.')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

cap = torch.cuda.get_device_capability()
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', cap)
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
assert cap >= (10, 0), f'v12 tensor-core MLA needs Blackwell (sm_100+); this GPU is {cap}. Rent a B300/sm_103a.'
if cap < (10, 3):
    print('NOTE: this is a dev rung (not sm_103). Arm 1 (FP8) builds; the record + Arm 2 (native NVFP4) run on B300/sm_103a.')


torch 2.10.0+cu128 | cuda 12.8 | cap (10, 3)
name, compute_cap, memory.total [MiB]
NVIDIA B300 SXM6 AC, 10.3, 275040 MiB
NVIDIA B300 SXM6 AC, 10.3, 275040 MiB
NVIDIA B300 SXM6 AC, 10.3, 275040 MiB
NVIDIA B300 SXM6 AC, 10.3, 275040 MiB


## 1. Get the repo

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())


Cloning into 'flashattention-cuda'...


Already up to date.
cwd /flashattention-cuda


From https://github.com/gkienpham-cmd/flashattention-cuda
 * branch            main       -> FETCH_HEAD


## 2. Roofline — the ENGINE sets the ridge (recorded BEFORE coding; results.md Step 12)

Same NVFP4 latent storage as v11 → **AI unchanged** (835 at h_q=128). The engine swap moves the **ridge**
(compute peak ÷ HBM BW). Under the *engine-correct* ridge: Arm 1 (FP8, ridge 625) is compute-bound ~1.34×;
**Arm 2 (NVFP4, ridge 1875) falls BACK to HBM-bound** — the 15 PF peak overshoots AI 835. The real
(per-CTA-corrected) prediction: single-token decode is SMEM-BW/pipeline-depth-bound either way.

In [3]:
from roofline.archs import get_arch
from roofline.model import estimate
L, R = 512, 64
for sm in ('sm_100', 'sm_103'):
    try:
        arch = get_arch(sm)
    except KeyError:
        continue
    print(f"\n=== {arch.name} ({sm}) | HBM {arch.hbm_bw_gbps} GB/s ===")
    print(f"  ridges (peak/BW): fp16-TC {arch.fp16_tc_flops/(arch.hbm_bw_gbps*1e9):.0f} | "
          f"FP8-TC {arch.fp8_tc_flops/(arch.hbm_bw_gbps*1e9):.0f} | NVFP4-TC {arch.fp4_tc_flops/(arch.hbm_bw_gbps*1e9):.0f}")
    print(f"  {'engine':>8} | {'AI':>7} | {'ridge':>6} | {'AI/ridge':>8} | {'limiter':>8}")
    for eng in ('fp16', 'fp8', 'nvfp4'):
        e = estimate(arch, B=1, H=1, N_q=1, N_k=8192, d=L+R, precision='nvfp4',
                     mla=True, h_q=128, kv_lora_rank=L, rope_dim=R, mma_engine=eng)
        print(f"  {eng:>8} | {e.arithmetic_intensity:7.1f} | {e.ridge:6.0f} | "
              f"{e.arithmetic_intensity/e.ridge:8.2f} | {e.limiter.upper():>8}")
print('\nPREDICTION (results.md Step 12): Arm 1 FP8 -> compute-bound 1.34x; Arm 2 NVFP4 -> HBM-bound (overshoot)')
print('+ exp (measured 0.5x) is the #2 term on Arm 2. REAL (per-CTA-corrected): SMEM-BW/pipeline-depth-bound,')
print('realized TFLOP/s << FP4 peak, will NOT beat FlashMLA ~410 TFLOP/s. COUNTER (the prize): if achieved')
print('TFLOP/s climbs >~10x v11 0.75 with high %SMEM-BW, the limiter LEFT per-CTA (first in the arc).')



=== NVIDIA B200 (Blackwell) (sm_100) | HBM 8000.0 GB/s ===
  ridges (peak/BW): fp16-TC 281 | FP8-TC 562 | NVFP4-TC 1125
    engine |      AI |  ridge | AI/ridge |  limiter
      fp16 |   835.0 |    281 |     2.97 |      MMA
       fp8 |   835.0 |    562 |     1.48 |      MMA
     nvfp4 |   835.0 |   1125 |     0.74 |      HBM

=== NVIDIA B300 (Blackwell Ultra, GB300) (sm_103) | HBM 8000.0 GB/s ===
  ridges (peak/BW): fp16-TC 312 | FP8-TC 625 | NVFP4-TC 1875
    engine |      AI |  ridge | AI/ridge |  limiter
      fp16 |   835.0 |    312 |     2.67 |      MMA
       fp8 |   835.0 |    625 |     1.34 |      MMA
     nvfp4 |   835.0 |   1875 |     0.45 |      HBM

PREDICTION (results.md Step 12): Arm 1 FP8 -> compute-bound 1.34x; Arm 2 NVFP4 -> HBM-bound (overshoot)
+ exp (measured 0.5x) is the #2 term on Arm 2. REAL (per-CTA-corrected): SMEM-BW/pipeline-depth-bound,
realized TFLOP/s << FP4 peak, will NOT beat FlashMLA ~410 TFLOP/s. COUNTER (the prize): if achieved
TFLOP/s climbs >~10x 

## 3. Build v12_mla_tc (JIT) — fork CUTLASS example 77

The real build needs **CUTLASS 4.x** include paths and **sm_103a** for Arm 2 (native NVFP4). Set
`FA_CUDA_ARCH=103a` and add the CUTLASS includes (see `bindings/load.py` `_ARCH` note). The scaffold
compiles without CUTLASS (plain sm_103) so wiring/build is testable — implement the tcgen05 body in
`kernels/v12_mla_tc/mla_tc_attention.cu` (header §"THE BUILD") before §4 can pass.

In [4]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v12_mla_tc')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
# For the real native-NVFP4 (Arm 2) build, uncomment:
# os.environ['FA_CUDA_ARCH'] = '103a'
# and add CUTLASS includes to bindings/load.py build_kernel (extra_include_paths) or CPLUS_INCLUDE_PATH.
from bindings.load import build_kernel
mla_tc = build_kernel('v12_mla_tc'); print('built v12 (tensor-core MLA):', mla_tc)


[1/3] c++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fa_v12_mla_tc -DTORCH_API_INCLUDE_EXTENSION_H -I/venv/main/lib/python3.12/site-packages/nvidia/cublas/include -I/venv/main/lib/python3.12/site-packages/nvidia/cuda_cupti/include -I/venv/main/lib/python3.12/site-packages/nvidia/cuda_nvrtc/include -I/venv/main/lib/python3.12/site-packages/nvidia/cuda_runtime/include -I/venv/main/lib/python3.12/site-packages/nvidia/cudnn/include -I/venv/main/lib/python3.12/site-packages/nvidia/cufft/include -I/venv/main/lib/python3.12/site-packages/nvidia/cufile/include -I/venv/main/lib/python3.12/site-packages/nvidia/curand/include -I/venv/main/lib/python3.12/site-packages/nvidia/cusolver/include -I/venv/main/lib/python3.12/site-packages/nvidia/cusparse/include -I/venv/main/lib/python3.12/site-packages/nvidia/cusparselt/include -I/venv/main/lib/python3.12/site-packages/nvidia/nccl/include -I/venv/main/lib/python3.12/site-packages/nvidia/nvjitlink/include -I/venv/main/lib/python3.12/site-packages/nvidi

## 4. Correctness gate — v12_mla_tc + v11 regression (Gate 1 of 2)

Apples-to-apples vs the SAME v11 oracle (`sdpa_reference_mla`) at tol 5e-2, both arms (the `nvfp4` arm
only at h_q≥128), plus the absorption identity (§9 Q4) and a v11 regression. Fails loudly until the
tcgen05 body lands.

In [5]:
!python -m pytest tests/test_correctness.py -k "v12_mla_tc or v11_mla" -q


........................................FsFsFsFsFFFFFsFsFsFsFFFFFsFsFsFs [ 76%]
FFFFFsFsFsFsFFFFFFFFFF                                                   [100%]
=================================== FAILURES ===================================
_________________ test_v12_mla_tc_decode[9-False-16-4096-fp8] __________________

seed = 9, causal = False, h_q = 16, N_k = 4096, engine = 'fp8'

    @requires_capability(10, 0)
    @pytest.mark.parametrize("engine", ["fp8", "nvfp4"])
    @pytest.mark.parametrize("N_k", [4096, 8190])              # 8190 -> non-multiple last page/split
    @pytest.mark.parametrize("h_q", [16, 64, 128])            # M-packing range; nvfp4 arm needs h_q>=128
    @pytest.mark.parametrize("causal", [False, True])
    @pytest.mark.parametrize("seed", [9, 17])
    def test_v12_mla_tc_decode(seed, causal, h_q, N_k, engine):
        if engine == "nvfp4" and h_q < 128:
            pytest.skip("native NVFP4 MMA (Arm 2) gates M=h_q>=128 (tcgen05 block-scaled)")
        torch.ma

## 5. The engine A/B — v12 vs v11 on the SAME latent bytes (same-session, clock-matched)

`vs naive` column = **vs v11** (CUDA-core MLA) — the trustworthy engine-isolation number. The pre-registered
prediction: v12 closes v11's 4× self-gap *toward* the TC ceiling but stays SMEM-BW/pipeline-depth-bound
(realized TFLOP/s << peak). Run both arms. **Lock clocks first** (root) to honor the v11 debt.

In [ ]:
# Optional clock lock (root) — pay the v11 debt. Falls back with a warning if unprivileged.
from bench.regime import lock_clocks, reset_clocks
locked, *_ = lock_clocks()
try:
    from bench.harness import run
    for eng in ('fp8', 'nvfp4'):
        print(f"\n##### v12 engine={eng} (real DeepSeek-V3 latent 576/512, h_q=128) #####")
        run('v12_mla_tc', 'fp16', batches=[1], H=128, causal=False, decode=True,
            seq_lens=[8192, 32768], head_dims=[576], mla_engine=eng)
finally:
    if locked:
        reset_clocks()


## 6. Regime sweep past L2 — does the engine leave per-CTA? (clock-locked, L2-flushed, the §9 Q2 deliverable)

`%HBM`/`eff_bw` vs N_k past the 132.6 MB B300 L2 (WS overflow by construction), both arms. The achieved
TFLOP/s + %SMEM-BW (from ncu, §8) is what settles compute-vs-SMEM-BW. Counter-free proxy carries here;
ncu validates it.

In [ ]:
# Clock-locked, L2-flushed past-L2 sweep (the T1/Q2 crossover). Needs root for --lock-clocks.
!python -m bench.regime --backend v12_mla_tc --engine nvfp4 --dim 576 --h-kv 1 --gqa-group 128 \
    --kv-lens 8192 32768 131072 524288 --batch 1 --lock-clocks
print('\n--- Arm 1 (FP8) for comparison ---')
!python -m bench.regime --backend v12_mla_tc --engine fp8 --dim 576 --h-kv 1 --gqa-group 128 \
    --kv-lens 8192 32768 131072 524288 --batch 1 --lock-clocks


## 7. [debts] ncu (privileged), nsys schedule, torch-baseline profile, 2×-exp re-ablation at M=128

The four v11 honesty debts, paid here. ncu needs a privileged/bare-metal box (unprivileged containers
return `ERR_NVGPUCTRPERM`). nsys needs **2025.3.2+** (older CUPTI records an empty sm_103 trace).

In [ ]:
# Debt #3 (RUNNABLE): the fp16 dense-MQA torch baseline — convert v11's "cuBLAS-TC-GEMM 4x cause" from
# INFERENCE to MEASUREMENT. v11's baseline was an FP32 batched-GEMV (M=1); this one is fp16, so cuBLAS
# routes the M=h_q=128 head-pack into a TENSOR-CORE GEMM (exactly the path that beat v11 ~4x). Time it
# here as the v12 "vs dense-MQA" comparator, then profile `dense_mqa` under nsys/ncu (cells below) to
# CONFIRM the cuBLAS TC-GEMM dispatch -> debt #3 paid (cause measured, not inferred).
import torch

def _time_ms(fn, warmup=10, iters=50):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
    ts = []
    for _ in range(iters):
        s.record(); fn(); e.record(); torch.cuda.synchronize(); ts.append(s.elapsed_time(e))
    ts.sort(); return ts[len(ts) // 2]

B, h_q, L, R = 1, 128, 512, 64          # real DeepSeek-V3 latent: DQK=576, DV=512, M=h_q=128
DQK, DV = L + R, L
scale = 1.0 / (DQK ** 0.5)
print(f"{'N_k':>8} | {'dense-MQA fp16 ms':>18} | {'us/tok':>8}")
for N_k in (8192, 32768, 131072):
    q   = torch.randn(B, h_q, 1, DQK, device='cuda', dtype=torch.float16)   # q_absorbed (latent basis)
    lat = torch.randn(B, 1, N_k, DQK, device='cuda', dtype=torch.float16)   # ONE shared latent head
    def dense_mqa(q=q, lat=lat):
        # MQA over the latent, fp16 in -> the two matmuls hit cuBLAS tensor cores (single latent head
        # broadcasts over all h_q query heads). Softmax in fp32 for stability, back to fp16 for the PV GEMM.
        s = torch.matmul(q, lat.transpose(-1, -2)) * scale                 # [B,h_q,1,N_k]  (QK, TC GEMM)
        p = torch.softmax(s.float() - s.float().amax(-1, keepdim=True), -1).half()
        return torch.matmul(p, lat[..., :DV])                              # [B,h_q,1,DV]   (PV, TC GEMM)
    ms = _time_ms(dense_mqa)
    print(f"{N_k:>8} | {ms:18.3f} | {ms * 1e3 / (B * h_q):8.2f}")
    del q, lat; torch.cuda.empty_cache()

print("\nThis is the v12 'vs dense-MQA' comparator at fp16 (NOT v11's FP32 GEMV). Compare to v12's us/tok")
print("(cell 12). Then profile dense_mqa under nsys/ncu (next cell) to MEASURE that cuBLAS dispatches a")
print("tensor-core GEMM (e.g. a *sm90/sm100* cutlass/cublas *gemm* kernel name) -> the 4x cause is measured.")


In [ ]:
# (a) ncu — achieved TFLOP/s + SMEM-BW + tensor-pipe util on ONE shape (the Q2 deliverable).
#     Needs --cap-add / privileged. profile_one loops the kernel so ncu can attach.
# !ncu --set full --kernel-name 'regex:.*' -c 1 \
#     python -m bench.regime --backend v12_mla_tc --engine nvfp4 --dim 576 \
#     --profile 1,1,131072,576
#
# (b) nsys schedule (partial vs merge share). Requires nsys 2025.3.2+.
# !nsys profile -o v12_kern --stats=true python -m bench.regime --backend v12_mla_tc \
#     --engine nvfp4 --dim 576 --profile 1,1,131072,576
#
# (c) torch-baseline profile at fp16/bf16 (convert v11's "cuBLAS-TC 4x cause" inference -> measurement).
#     Profile the dense-MQA reference, NOT FP32 batched-GEMV.
#
# (d) 2x-exp re-ablation at M=128 (Q3): EX2 throughput on the M=128 score tile (vs v10/v11's M=1 0.5x).
print('Templates above — fill on the privileged B300/sm_103a box; save outputs alongside this notebook.')


## 8. Verdict (fill after the run)

- **Pure-roofline (engine-correct):** Arm 1 FP8 compute-bound 1.34×; Arm 2 NVFP4 HBM-bound (overshoot). [measured: …]
- **Per-CTA-corrected (the real prediction):** SMEM-BW/pipeline-depth-bound, realized TFLOP/s << peak, does NOT beat FlashMLA. [measured: …]
- **Counter-prediction:** did achieved TFLOP/s climb >~10× v11's 0.75 (limiter LEFT per-CTA)? [yes/no: …]
- **§9 Q1:** did M=128 pack as ONE tcgen05 GEMM (ex77 num_groups 32 vs 128)? [verified: …]
- **vs v11 (engine A/B):** [×]. **vs FlashMLA/FlashInfer:** [×] (complementing, not beating).
- **Debts paid:** ncu ✔/✗ · clock-lock ✔ · torch fp16 profile ✔ · 2×-exp@M=128 ✔.

Record into `docs/results.md`/`decisions.md` Step 12 + `interview-prep.md` C18, then the quiz.